# ⚡ TEMPLATE KILAT RESPONSI UTS - PRAKTIKUM BIG DATA
> Jalankan sel secara berurutan atau contek fungsi yang kamu butuhkan sesuai instruksi soal ujian!

## 1. Setup Environment & Inisialisasi SparkSession (Arch Linux Safe)

In [ ]:
import os
import sys
import getpass
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Konfigurasi isolasi lingkungan native Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \n    .appName("Responsi-UTS") \n    .master("local[*]") \n    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
username = os.environ.get("USER", getpass.getuser())
print(f"✅ SparkSession Aktif untuk pengguna: {username}")

## 2. EXTRACT — Membaca Data Mentah (HDFS / Lokal)

In [ ]:
# Ganti path berikut sesuai dengan lokasi file tugasmu di HDFS atau lokal
# hdfs_path = f"hdfs://localhost:9000/user/{username}/responsi/transaksi.csv"
# df_trx = spark.read.option("header", "true").option("inferSchema", "true").csv(hdfs_path)

# Dummy data contoh jika file belum ada di HDFS:
data_dummy = [
    ("TRX-001", "Magelang", "Elektronik", 2, 500000, 5),
    ("TRX-002", "Magelang", "Fashion", 5, 100000, None),
    ("TRX-003", "Yogyakarta", "Elektronik", 1, 1500000, 4),
    ("TRX-004", "Yogyakarta", "Makanan", 10, 25000, 5),
    ("TRX-005", "Semarang", "Elektronik", 3, 600000, None),
    ("TRX-006", "Semarang", "Fashion", 4, 150000, 3),
    ("TRX-007", "Purworejo", "Makanan", 20, 15000, 4),
    ("TRX-008", "Purworejo", "Elektronik", 1, 2000000, None),
]
cols = ["order_id", "kota", "kategori", "unit_terjual", "harga_satuan", "rating"]
df_trx = spark.createDataFrame(data_dummy, cols)
df_trx.show(5)

## 3. TRANSFORM — Operasi Kolom, Filter, & Handling Null

In [ ]:
# A. Tambah kolom total pendapatan
df_trx = df_trx.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga_satuan"))

# B. Tambah flag boolean deteksi apakah diulas
df_trx = df_trx.withColumn("ada_rating", F.col("rating").isNotNull())

# C. Imputasi nilai null (Sentinel Value 0)
df_bersih = df_trx.na.fill({"rating": 0})

# D. Filter multi-kondisi (Contoh: omset >= 500rb dan bukan Magelang)
df_filter = df_bersih.filter((F.col("total_pendapatan") >= 500000) & (F.col("kota") != "Magelang"))
df_filter.show()

## 4. TRANSFORM — Multi-Table Relational Join

In [ ]:
# Tabel referensi target cabang
target_data = [
    ("Magelang", "Budi Santoso", 3000000),
    ("Yogyakarta", "Siti Aminah", 5000000),
    ("Semarang", "Agus Setiawan", 4500000),
    ("Purworejo", "Dewi Lestari", 2500000),
]
df_target = spark.createDataFrame(target_data, ["kota", "pic_cabang", "target_omset"])

# Best Practice: Agregasi data transaksi besar dulu
df_omset_kota = df_bersih.groupBy("kota").agg(F.sum("total_pendapatan").alias("realisasi_omset"))

# Join dengan target
df_pencapaian = df_omset_kota.join(df_target, on="kota", how="inner") \n    .withColumn("persen_capaian", F.round((F.col("realisasi_omset") / F.col("target_omset")) * 100, 2)) \n    .orderBy(F.col("persen_capaian").desc())

df_pencapaian.show()

## 5. TRANSFORM — Window Function (Ranking & Top-N)

In [ ]:
# Rekap omset per kombinasi kota dan kategori
df_kota_kat = df_bersih.groupBy("kota", "kategori").agg(
    F.sum("total_pendapatan").alias("omset_kategori")
)

# Definisikan Window partisi kota urut omset menurun
window_kota = Window.partitionBy("kota").orderBy(F.col("omset_kategori").desc())

# Beri nomor urut dan filter juara 1
df_top1 = df_kota_kat \n    .withColumn("rank", F.row_number().over(window_kota)) \n    .filter(F.col("rank") == 1) \n    .orderBy(F.col("omset_kategori").desc())

df_top1.show()

## 6. TRANSFORM — Spark SQL (Kueri Relasional Murni)

In [ ]:
# Daftarkan view sementara
df_bersih.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("cabang")

df_sql = spark.sql("""
    SELECT 
        t.kota,
        c.pic_cabang,
        COUNT(t.order_id) AS total_transaksi,
        SUM(t.total_pendapatan) AS total_omset
    FROM transaksi t
    JOIN cabang c ON t.kota = c.kota
    GROUP BY t.kota, c.pic_cabang
    ORDER BY total_omset DESC
""")
df_sql.show()

## 7. LOAD — Menyimpan Hasil Akhir ke HDFS Parquet Terpartisi

In [ ]:
output_hdfs = f"hdfs://localhost:9000/user/{username}/responsi/processed/hasil_laporan"

df_bersih.write.mode("overwrite") \n    .partitionBy("kategori") \n    .parquet(output_hdfs)

print(f"✅ Berhasil disimpan ke HDFS: {output_hdfs}")

# Verifikasi pembacaan ulang
df_cek = spark.read.parquet(output_hdfs)
print(f"Jumlah baris di HDFS: {df_cek.count()}")

## 8. Selesai — Matikan SparkSession

In [ ]:
spark.stop()
print("🛑 SparkSession ditutup. Resource RAM dan CPU telah dibebaskan.")